In [ ]:
#Identify categorical columns and convert them into categories
#Convert columns to category dtype
def process_categorical(X):
    cat_columns = X.select_dtypes(include=['object']).columns.tolist() #your categorical columns
    for col in cat_columns:
        X[col] = X[col].astype('category')
        X[col] = X[col].astype('category')
    print(cat_columns)
    print(f"Found {len(cat_columns)} categorical columns")



In [ ]:
#Aim - compare efficacy of synthetic data on XGBoost
#Evaluate efficacy of regression model
#Do this by appending synthetic data to train data
#Set up train and test splits
import pandas as pd
from sklearn.model_selection import train_test_split
from sksurv.util import Surv
import numpy as np

df = pd.read_csv('../Resources/Overall_cleaned.csv',encoding='latin-1')
y_time = df['Survival months']
y_event = df['Vital status recode (study cutoff used)']
X = df.drop(columns=['Survival months',
                     'Vital status recode (study cutoff used)'])


#For testing feature interaction
# X['Tumor Size Adjusted'] = np.where(
#     X['Primary Site - labeled'] == 'C34.1-Upper lobe,lung',
#     X['Total number of in situ/malignant tumors for patient'] * 1,
#     np.where(
#         X['Primary Site - labeled'] == 'C34.2-Middle lobe',
#         X['Total number of in situ/malignant tumors for patient'] * 4,
#         np.where(
#             X['Primary Site - labeled'] == 'C34.3-Lower lobe,lung',
#             X['Total number of in situ/malignant tumors for patient'] * 2,
#             np.where(
#                 X['Primary Site - labeled'] == 'C34.9-Lung, NOS',
#                 X['Total number of in situ/malignant tumors for patient'] * 4,
#                 np.where(
#                     X['Primary Site - labeled'] == 'C34.0-Main bronchus',
#                     X['Total number of in situ/malignant tumors for patient'] * 5,
#                     X['Total number of in situ/malignant tumors for patient'] * 6
#                 )
#             )
#         )
#     )
# )

# X['Tumour Number Bone'] = np.where(
#     X['SEER Combined Mets at DX-bone (2010+)'] == "Yes"
#     , X['Total number of in situ/malignant tumors for patient'] * 1.5, X['Total number of in situ/malignant tumors for patient'] * 1)

#OPTIONAL - convert all columns to numerical 
for col in X.select_dtypes(include=['object', 'category']).columns:
    X[col] = pd.factorize(X[col])[0]

print(X.dtypes)
process_categorical(X)
#create survival target (used for train test split only)
y_struct = Surv.from_arrays(event=y_event,time=y_time)
X_train,X_test,y_train,y_test = train_test_split(X,y_struct,test_size=0.8,random_state=42)

In [ ]:
from matplotlib import pyplot as plt
dead_only_df = df[df['Vital status recode (study cutoff used)']==1]
alive_only_df = df[df['Vital status recode (study cutoff used)']==0]

#get distribution of survival months from dead patients
survival_dist = alive_only_df['Survival months'].value_counts()/len(alive_only_df)
cumulative_sum_first_50 = survival_dist.iloc[:50].sum()
print(f"Sum of proportions for the first 50 index positions: {cumulative_sum_first_50:.4f}")
plt.figure(figsize=(10, 6))
plt.plot(survival_dist.index, survival_dist.values, marker='o', linestyle='-')

In [ ]:
len(df)

In [ ]:
print(df['Primary Site - labeled'].value_counts())

In [ ]:
#show the categorical column with the highest number of unique values
cat_cols = df.select_dtypes(include=['object', 'category']).columns
nunique_cat = df[cat_cols].nunique()
max_unique = nunique_cat.max()
col_with_max_unique = nunique_cat[nunique_cat == max_unique].index.tolist()
print(f"Categorical column(s) with the most unique values ({max_unique}): {col_with_max_unique}")

In [ ]:
#convert to XGBoost survival format (negative time = censored)
y_train_surv = y_train['time'].copy()
y_train_surv[~y_train['event']] *= -1  #negative for censored

y_test_surv = y_test['time'].copy()
y_test_surv[~y_test['event']] *= -1

In [ ]:
#clean column names to avoid XGBoost errors
import re

# Clean column names - remove or replace invalid characters
X_train.columns = [re.sub(r'[\[\]<>]', '_', str(col)) for col in X_train.columns]
X_test.columns = [re.sub(r'[\[\]<>]', '_', str(col)) for col in X_test.columns]

In [ ]:
import xgboost as xgb
from lifelines.utils import concordance_index

model = xgb.XGBRegressor(
    objective='survival:cox',
    eval_metric='cox-nloglik',
    tree_method='hist',
    enable_categorical=True
)

model.fit(X_train, y_train_surv)

In [ ]:
apply_bootrstrapping = False #OPTIONAL: APPLY BOOTSTRAPPING

import numpy as np
from scipy.stats import boxcox
import matplotlib.pyplot as plt

if apply_bootrstrapping:
    n = len(X_train)
    n_bootstrap_sets = 5
    models = [] #length will be equal to n_bootstrap_sets

    #go through each bootstrap subset, fitting a new model (one model per bootstrap subset)
    for i in range(n_bootstrap_sets):
        #Sample with replacement (bootstrap)
        indices = np.random.choice(n, size=n, replace=True)
        X_boot = X_train.iloc[indices].reset_index(drop=True)
        y_boot = y_train[indices]
        process_categorical(X_boot)
        #train a model on the bootstrapped subset
        model = xgb.XGBRegressor(
        objective='survival:cox',
        eval_metric='cox-nloglik',
        tree_method='hist',
        enable_categorical=True
        )
        y_boot_surv = y_boot['time'].copy()
        y_boot_surv[~y_boot['event']] *= -1  # Negative for censored
        model.fit(X_boot, y_boot_surv)

        models.append(model)

    results = [model.predict(X_train) for model in models]
    
    #average survival curves across all bootstrap models
    train_scores = sum(results) / len(results)
    #apply transformation to make data more normally distributed
    train_scores_bc,best_lambda = boxcox(train_scores)
else:
    #get risk scores on training data
    train_scores = model.predict(X_train)
    print(pd.DataFrame({'train_scores': train_scores}).value_counts())
    #Box cox finds optimal lambda
    train_scores_bc,best_lambda = boxcox(train_scores)

#for generating risk labels
#create 10 equal-width bins based on training range
min_score, max_score = train_scores_bc.min(), train_scores_bc.max()
bins = np.linspace(min_score, max_score, 5) #5 edges = 4 bins

In [ ]:
#apply to labels to test data
if apply_bootrstrapping:
    
    process_categorical(X_test)
    results = [model.predict(X_test) for model in models]
    #average survival curves across all bootstrap models
    test_scores = sum(results) / len(results)
    #apply transformation to make data more normally distributed
    test_scores_bc,best_lambda = boxcox(test_scores)
else:
    test_scores = model.predict(X_test)
    test_scores_bc, best_lambda = boxcox(test_scores)

test_risk_groups = pd.cut(test_scores_bc, bins=bins, labels=range(1, 5), include_lowest=True)

# Evaluate
# Calculate C-index (higher is better, 0.5 = random, 1.0 = perfect)
c_index = concordance_index(y_test['time'], -test_scores, y_test['event'])
print(f"C-index: {c_index:.4f}")

In [ ]:
#Make KM plot
import sys
import importlib

sys.path.append('..')
# Remove cached module
if 'KM_Plotter' in sys.modules:
    del sys.modules['KM_Plotter']

# Reload the module
from Classes.KM_Plotter import KM_Plotter
importlib.reload(sys.modules['Classes.KM_Plotter'])

X_test_extended = X_test.copy()
X_test_extended = pd.DataFrame(X_test,columns=X_test.columns)
X_test_extended['Risk_Label'] = test_risk_groups.map(lambda x: f'EL{x}')
X_test_extended['Survival_Time'] = y_test['time']
X_test_extended['Event_Status'] = y_test['event']

km_plotter = KM_Plotter(X_test_extended)
km_plotter.makeKMPlot()

In [ ]:
import xgboost as xgb

model = xgb.XGBRegressor(
            objective='survival:cox',
            eval_metric='cox-nloglik',
            tree_method='hist',
            enable_categorical=True,
        )
model.fit(X_train, y_train_surv)

In [ ]:
print(X.columns)
#to get parameter configuration for 'monotone_constraints'
constraints_lst = []
target_cols = ['Age recode with <1 year olds and 90+','Tumor Size Summary (2016+)','Regional nodes positive (1988+)']
for col in X.columns:
    if col in target_cols:
        constraints_lst.append(1)
    else:
        constraints_lst.append(0)
print(constraints_lst)

In [ ]:
import skopt.space.space as s

search_space = list()

search_space.append(s.Real(0.01,0.9,name='eta'))
search_space.append(s.Categorical(['gbtree','gblinear'],name='booster'))
search_space.append(s.Real(0.5,0.8,name='col_sample_bylevel'))
search_space.append(s.Real(0.5,0.7,name='colsample_bytree'))
search_space.append(s.Integer(1,42,name='max_cat_threshold'))
search_space.append(s.Real(1,10,name='max_delta_step'))
search_space.append(s.Real(1,5,name='gamma'))
search_space.append(s.Integer(2,6,name='max_depth'))  # reduced upper bound
search_space.append(s.Integer(2,32,name='max_leaves'))  # reduced upper bound
search_space.append(s.Integer(1,10,name='min_child_weight'))
search_space.append(s.Integer(50,300,name='n_estimators'))  # reduced upper bound
search_space.append(s.Integer(1,10,name='num_parallel_tree'))
search_space.append(s.Real(0,10,name='reg_alpha'))
search_space.append(s.Real(0,10,name='reg_lambda'))
search_space.append(s.Real(1,10,name='scale_pos_weight'))
search_space.append(s.Real(0.5,1,name='subsample'))
search_space.append(s.Categorical(['auto','approx','hist'],name='tree_method'))

In [ ]:
#gives precision score
from sklearn.model_selection import cross_val_score
#for cross validation
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.svm import SVC
from numpy import mean
from numpy import std
import numpy as np
import sys
import importlib
import xgboost as xgb
sys.path.append('..')
from Classes.KM_Plotter import KM_Plotter
importlib.reload(sys.modules['Classes.KM_Plotter'])

def evaluate_model(params):
    
    eta, booster, colsample_bylevel, colsample_bytree, max_cat_threshold, max_delta_step, gamma, max_depth, max_leaves, min_child_weight, n_estimators, num_parallel_tree, reg_alpha, reg_lambda, scale_pos_weight, subsample, tree_method = params

    print("Evaluating parameters:")
    print({
        'eta': eta,
        'booster': booster,
        'colsample_bylevel': colsample_bylevel,
        'colsample_bytree': colsample_bytree,
        'max_cat_threshold': max_cat_threshold,
        'max_delta_step': max_delta_step,
        'gamma': gamma,
        'max_depth': max_depth,
        'max_leaves': max_leaves,
        'min_child_weight': min_child_weight,
        'n_estimators': n_estimators,
        'num_parallel_tree': num_parallel_tree,
        'reg_alpha': reg_alpha,
        'reg_lambda': reg_lambda,
        'scale_pos_weight': scale_pos_weight,
        'subsample': subsample,
        'tree_method': tree_method
    })

    model = xgb.XGBRegressor(
        objective='survival:cox',
        eval_metric='cox-nloglik',
        base_score=0.5,
        max_bin=296,
        enable_categorical=True,
        grow_policy = 'depthwise',
        eta = eta,
        colsample_bylevel = colsample_bylevel,
        #output from constraints_lst from cell above
        monotone_constraints = (0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0),
        booster = booster,
        max_cat_threshold = max_cat_threshold,
        max_delta_step = max_delta_step,
        gamma = gamma,
        max_depth = max_depth,
        max_leaves = max_leaves,
        min_child_weight = min_child_weight,
        n_estimators = n_estimators,
        num_parallel_tree = num_parallel_tree,
        reg_alpha = reg_alpha,
        reg_lambda = reg_lambda,
        sampling_method = 'uniform',
        scale_pos_weight = scale_pos_weight,
        subsample = subsample,
        tree_method = tree_method,
        colsample_bytree = colsample_bytree
    )
    num_curves = 6
    #define the splits
    cv = RepeatedStratifiedKFold(n_splits=2,n_repeats=1,random_state=1)
    count=0
    scores = []
    #apply the splits 
    for train_idx, test_idx in cv.split(X_train,y_train):
        count+=1 #check that it's 30 iterations
        print(f"count is {count}")
        subset_X_train = X_train.iloc[train_idx]
        subset_y_train_surv = y_train_surv[train_idx]
        subset_X_test = X_train.iloc[test_idx]
        subset_y_test = y_train[test_idx] #we don't want surv since that is just one array
        

        model.fit(subset_X_train, subset_y_train_surv)
        model_scores = model.predict(subset_X_train)
        #create 3 equal width bins based on training range
        min_score, max_score = model_scores.min(), model_scores.max()
        bins = np.linspace(min_score, max_score, num_curves+1) #4 edges = 3 bins
        bins[0] = -np.inf
        bins[-1] = np.inf
        #bins = np.quantile(model_scores, q=[0, 0.5, 1.0])
        
        #assign the test samples to corresponding bins, thus assigning labels
        subset_test_scores = model.predict(subset_X_test)
        print(f"Bins are: {bins}")

        try:
            subset_test_risk_groups = pd.cut(subset_test_scores, bins=bins, labels=range(1, num_curves+1), include_lowest=True)
        except Exception as e:
            print(f"Error in binning: {e}")
            print(f"bins were: {bins}")
            scores.append(1000)
            continue 

        subset_X_test_extended = subset_X_test.copy()
        subset_X_test_extended = pd.DataFrame(subset_X_test,columns=X_test.columns)
        subset_X_test_extended['Risk_Label'] = subset_test_risk_groups.map(lambda x: f'EL{x}')
        subset_X_test_extended['Survival_Time'] = subset_y_test['time']
        subset_X_test_extended['Event_Status'] = subset_y_test['event']

        km_plotter = KM_Plotter(subset_X_test_extended,num_curves=num_curves)
        km_plotter.makeKMPlot(plot=True)
        score = km_plotter.getKMScore()
        scores.append(score)
        print(f"Score for fold is: {score}")
    return(mean(scores))

In [ ]:
#gives precision score
from sklearn.model_selection import cross_val_score
#for cross validation
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.svm import SVC
from numpy import mean
from numpy import std
import numpy as np
import sys
import importlib
import xgboost as xgb
sys.path.append('..')
from Classes.KM_Plotter import KM_Plotter
importlib.reload(sys.modules['Classes.KM_Plotter'])

def evaluate_model(params, X_train_subset, y_train_subset, y_train_surv_subset, X_val, y_val):
    eta, booster, colsample_bylevel, colsample_bytree, max_cat_threshold, max_delta_step, gamma, max_depth, max_leaves, min_child_weight, n_estimators, num_parallel_tree, reg_alpha, reg_lambda, scale_pos_weight, subsample, tree_method = params
    
    print("Evaluating parameters:")
    print({
        'eta': eta,
        'booster': booster,
        'colsample_bylevel': colsample_bylevel,
        'colsample_bytree': colsample_bytree,
        'max_cat_threshold': max_cat_threshold,
        'max_delta_step': max_delta_step,
        'gamma': gamma,
        'max_depth': max_depth,
        'max_leaves': max_leaves,
        'min_child_weight': min_child_weight,
        'n_estimators': n_estimators,
        'num_parallel_tree': num_parallel_tree,
        'reg_alpha': reg_alpha,
        'reg_lambda': reg_lambda,
        'scale_pos_weight': scale_pos_weight,
        'subsample': subsample,
        'tree_method': tree_method
    })
    
    model = xgb.XGBRegressor(
        objective='survival:cox',
        eval_metric='cox-nloglik',
        base_score=0.5,
        max_bin=296,
        enable_categorical=True,
        grow_policy = 'depthwise',
        eta = eta,
        colsample_bylevel = colsample_bylevel,
        #output from constraints_lst from cell above
        monotone_constraints = (0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0),
        booster = booster,
        max_cat_threshold = max_cat_threshold,
        max_delta_step = max_delta_step,
        gamma = gamma,
        max_depth = max_depth,
        max_leaves = max_leaves,
        min_child_weight = min_child_weight,
        n_estimators = n_estimators,
        num_parallel_tree = num_parallel_tree,
        reg_alpha = reg_alpha,
        reg_lambda = reg_lambda,
        sampling_method = 'uniform',
        scale_pos_weight = scale_pos_weight,
        subsample = subsample,
        tree_method = tree_method,
        colsample_bytree = colsample_bytree
    )
    num_curves = 6
    
    model.fit(X_train_subset, y_train_surv_subset)
    model_scores = model.predict(X_train_subset)
    #create 3 equal width bins based on training range
    min_score, max_score = model_scores.min(), model_scores.max()
    bins = np.linspace(min_score, max_score, num_curves+1) #4 edges = 3 bins
    bins[0] = -np.inf
    bins[-1] = np.inf
    
    #assign the validation samples to corresponding bins, thus assigning labels
    val_scores = model.predict(X_val)
    print(f"Bins are: {bins}")
    
    try:
        val_risk_groups = pd.cut(val_scores, bins=bins, labels=range(1, num_curves+1), include_lowest=True)
    except Exception as e:
        print(f"Error in binning: {e}")
        print(f"bins were: {bins}")
        return 1000
    
    X_val_extended = pd.DataFrame(X_val, columns=X_train.columns)
    X_val_extended['Risk_Label'] = val_risk_groups.map(lambda x: f'EL{x}')
    X_val_extended['Survival_Time'] = y_val['time']
    X_val_extended['Event_Status'] = y_val['event']
    
    km_plotter = KM_Plotter(X_val_extended, num_curves=num_curves)
    km_plotter.makeKMPlot(plot=True)
    score = km_plotter.getKMScore()
    print(f"Score for validation fold: {score}")
    return score

In [ ]:
import skopt

cv = RepeatedStratifiedKFold(n_splits=2,n_repeats=1,random_state=1)
best_scores = []
best_configs = [] 
#apply the splits 
for train_idx, test_idx in cv.split(X_train,y_train):
    count+=1 #check that it's 30 iterations
    print(f"count is {count}")
    X_train_subset = X_train.iloc[train_idx]
    y_train_subset = y_train[train_idx]
    y_train_surv_subset = y_train_surv[train_idx]
    X_val = X_train.iloc[test_idx]
    y_val = y_train[test_idx] #we don't want surv since that is just one array

    result = skopt.gp_minimize(
        lambda params: evaluate_model(params, X_train_subset, y_train_subset, y_train_surv_subset, X_val, y_val),
        search_space,
        n_jobs=3,
        n_calls=50,
        random_state=42
    )
    best_scores.append(result.fun)
    best_configs.append(result.x)


In [ ]:
print(best_scores)
print(best_configs
)

In [ ]:
print(f"Lowest score is: {result.fun}")
print(f"Best parameters: {result.x}")

In [ ]:
c_index = concordance_index(y_test['time'], -test_scores, y_test['event'])
print(f"C-index: {c_index:.4f}")

In [ ]:
#Visualise scores from best model 
import xgboost as xgb
from lifelines.utils import concordance_index
sys.path.append('..')
from Classes.KM_Plotter import KM_Plotter
importlib.reload(sys.modules['Classes.KM_Plotter'])

eta, booster, colsample_bylevel, colsample_bytree, max_cat_threshold, max_delta_step, gamma, max_depth, max_leaves, min_child_weight, n_estimators, num_parallel_tree, reg_alpha, reg_lambda, scale_pos_weight, subsample, tree_method = (0.4085270164700517, 'gblinear', 0.6778090171638181, 0.5161706652665431, np.int64(16), 3.1794394444968335, 4.212559025519584, np.int64(4), np.int64(32), np.int64(5), np.int64(254), np.int64(8), 1.507175439654295, 5.081987767407188, 7.262315261117938, 0.92917940240686, 'auto')

num_curves = 6

model = xgb.XGBRegressor(
    objective='survival:cox',
        eval_metric='cox-nloglik',
        base_score=0.5,
        max_bin=296,
        enable_categorical=True,
        grow_policy = 'depthwise',
        eta = eta,
        colsample_bylevel = colsample_bylevel,
        #output from constraints_lst from cell above
        monotone_constraints = (0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0),
        booster = booster,
        max_cat_threshold = max_cat_threshold,
        max_delta_step = max_delta_step,
        gamma = gamma,
        max_depth = max_depth,
        max_leaves = max_leaves,
        min_child_weight = min_child_weight,
        n_estimators = n_estimators,
        num_parallel_tree = num_parallel_tree,
        reg_alpha = reg_alpha,
        reg_lambda = reg_lambda,
        sampling_method = 'uniform',
        scale_pos_weight = scale_pos_weight,
        subsample = subsample,
        tree_method = tree_method,
        colsample_bytree = colsample_bytree
)

model.fit(X_train, y_train_surv)

model_scores = model.predict(X_train)
#create 3 equal width bins based on training range
min_score, max_score = model_scores.min(), model_scores.max()
bins = np.linspace(min_score, max_score, num_curves+1) #num_curves+1 edges = num_curves bins
bins[0] = -np.inf
bins[-1] = np.inf

test_scores = model.predict(X_test)

test_risk_groups = pd.cut(test_scores, bins=bins, labels=range(1, num_curves+1), include_lowest=True)

#Make KM plot
import sys
import importlib

sys.path.append('..')
# Remove cached module
if 'KM_Plotter' in sys.modules:
    del sys.modules['KM_Plotter']

# Reload the module
from Classes.KM_Plotter import KM_Plotter
importlib.reload(sys.modules['Classes.KM_Plotter'])

X_test_extended = X_test.copy()
X_test_extended = pd.DataFrame(X_test,columns=X_test.columns)
X_test_extended['Risk_Label'] = test_risk_groups.map(lambda x: f'EL{x}')
X_test_extended['Survival_Time'] = y_test['time']
X_test_extended['Event_Status'] = y_test['event']

km_plotter = KM_Plotter(X_test_extended, num_curves=num_curves)
km_plotter.makeKMPlot()
print(km_plotter.getKMScore())

In [ ]:
#Convert risk scores into survival probabilities

def get_baseline_cumulative_hazard_function(X_train,y_train,t):
    #t_k - unique observed event times
    event_mask = (y_train['event'] == True) & (y_train['time'] < t)
    event_times = y_train['time'][event_mask]

    hazard_ratio = model.predict(X_train)

    t_k_list,d_k_list = np.unique(event_times,return_counts=True)

    big_sum = 0
    for i in range(len(t_k_list)):
        risk_set_mask = y_train['time'] > t_k_list[i]
        big_sum = big_sum + (d_k_list[i]/np.sum(hazard_ratio[risk_set_mask]))
    return big_sum


#survival function
def get_survival_probabilities(t,X_train,X_test,y_train):
    #sequence representing rectangle widths in reimann integration
    
    #for each time point, get baseline cumulative hazard function
  
    #multiply reimann integral by hazard_ratio
    baseline_cumulative_hazard_function = get_baseline_cumulative_hazard_function(X_train,y_train,t)
    survival_probabilities = np.exp(-(model.predict(X_test))*baseline_cumulative_hazard_function)
    return survival_probabilities


survival_probabilities = get_survival_probabilities(5,X_train,X_test,y_train)


In [ ]:
print(survival_probabilities)
print(y_test['time'])
print(y_test['event']) #1 means dead

In [ ]:
#Get D calibration
from Classes.D_Calibration import D_Calibration

d_calibrator = D_Calibration(X_test,y_test['time'])
print(d_calibrator.D_calibration(survival_probabilities))

In [ ]:
# Get the number of unique values for each column
unique_counts = df.nunique()

# Find the maximum number of unique values
max_unique = unique_counts.max()

# Get the column(s) with the maximum number of unique values
columns_with_max_unique = unique_counts[unique_counts == max_unique].index.tolist()

print(f"Column(s) with the most unique values ({max_unique}): {columns_with_max_unique}")

In [ ]:
#Evaluate parameter importance for XGBoost model
importance = model.get_booster().get_score(importance_type='gain')
importance_df = pd.DataFrame({
    'feature': importance.keys(),
    'importance': importance.values()
}).sort_values('importance', ascending=False)

print(importance_df)

# Plot
plt.figure(figsize=(10, 8))
plt.barh(importance_df['feature'], importance_df['importance'])
plt.xlabel('Gain')
plt.title('Feature Importance (Gain)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

#give GPT some samples of the data and ask it for more techniques to investigate it further
#is there other things we can predict, that can influence survival months
#feature engineering requires luck 
#look at a way to explore several features - investigate importance of each column
#look at how importance can be useful - then give suggestions on how a person can improve it 

#come to the next meeting with suggestions on how to get findings on data and explore the features
#p values change based on different conditions - so many statistical values to show factors
#even if we have good p values, is it really that significant
#find what to solve
#understand XGBoost


In [ ]:
fig, ax = plt.subplots(figsize=(50, 30))
xgb.plot_tree(model, num_trees=50, ax=ax)
plt.savefig('../Resources/tree.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# TODO: Fix
 
# #Get features used in splits and all the feature names the model knows
# model.get_booster().get_score(importance_type='weight')

# # Get tree structure as dataframe
# tree_df = model.get_booster().trees_to_dataframe()
# print(tree_df)

# # Unique features used
# print(tree_df['Feature'].unique())

In [ ]:
from sksurv.metrics import integrated_brier_score

time_points = [10,20,30,40,50,60,70,80,90,100,110,120]
probabilities = [] #array of arrays (one for each time point)
for time_point in time_points:
    probabilities.append(get_survival_probabilities(time_point,X_train,X_test,y_train))
probabilities = np.column_stack(probabilities)

score = integrated_brier_score(y_train,y_test,probabilities,time_points)
print(score)

In [ ]:
clean_df = pd.read_csv('../Resources/Overall_cleaned.csv',encoding='latin-1')
clean_df[:100].to_csv('../Resources/Overall_Cleaned_Visual.csv', index=False)

raw_df = pd.read_csv('../Resources/Overall.csv',encoding='latin-1')
raw_df[:100].to_csv('../Resources/Overall_Raw_Visual.csv', index=False)


In [ ]:
import shap

shap.initjs()

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train)

In [ ]:
# Option 2: Normalize features to 0-1 range for consistent coloring
from sklearn.preprocessing import MinMaxScaler
import importlib
import shap.plots._beeswarm as beeswarm
importlib.reload(beeswarm)
from matplotlib.colors import LinearSegmentedColormap


shap.summary_plot(shap_values, X_train)

#Understand which features matter most overall
#shap.summary_plot(shap_values,X_train)
#Understand reasoning behind model, what are the features impacting model predictions
#Shapley sorts features based on their predictive power
#Every dot denotes record

In [ ]:
# Compare SHAP values by sex group
df_shap_values = pd.DataFrame(shap_values)
# Then test if difference is statistically significant
df_shap_values

In [ ]:
#booster object - handles training, prediction etc
bst = model.get_booster()

#return list of strings where each string contains structure of each decision tree
for line in bst.get_dump():
    if "Regional nodes positive" in line:
        print(line)

# 2. or, nicer, turn the whole forest into a DataFrame
df = bst.trees_to_dataframe()

# show only the rows where our feature is used
mask = df['Feature'].str.contains('Regional nodes positive', na=False)
df[mask][['Tree','Node','Feature','Split','Yes','No','Missing']]

# Histogram of split values for Regional Nodes Positive
import matplotlib.pyplot as plt

split_values = df[mask]['Split']

plt.figure(figsize=(10, 6))
plt.hist(split_values, bins=100, edgecolor='black', alpha=0.7)
plt.xlabel('Split Values')
plt.ylabel('Frequency')
plt.title('Histogram of Split Values for Regional Nodes Positive')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Find the column name (may have cleaned characters)
col_name = [c for c in X_train.columns if 'Regional nodes positive' in c][0]

plt.figure(figsize=(10, 6))
plt.hist(X_train[col_name], bins=200, edgecolor='black', alpha=0.7)
plt.xlabel('Regional Nodes Positive')
plt.ylabel('Frequency')
plt.title('Distribution of Regional Nodes Positive')
plt.show()


In [ ]:
#Why did patient X get this prediction - waterfall plot
shap.plots.waterfall(shap.Explanation(values=shap_values[2], 
                                       base_values=explainer.expected_value, 
                                       data=X_train.iloc[2]))

In [ ]:
def plot_boxplot_with_stats(column_name):
      col_name = [c for c in X_train.columns if column_name in c][0]
      col_df = pd.concat([X_train[col_name], X_test[col_name]]).dropna()

      # Calculate quartile statistics
      Q1 = col_df.quantile(0.25)
      median = col_df.median()
      Q3 = col_df.quantile(0.75)
      IQR = Q3 - Q1
      lower_whisker = col_df.min()
      upper_whisker = col_df.max()
      outliers = col_df[(col_df < Q1 - 1.5*IQR) | (col_df > Q3 + 1.5*IQR)]

      # Create boxplot
      fig, ax = plt.subplots(figsize=(9, 6))
      bp = ax.boxplot(col_df, vert=False, patch_artist=True, widths=0.5)

      medians = [item.get_xdata()[0] for item in bp['medians']]
      print(f'Medians: {medians}')

      minimums = [item.get_xdata()[0] for item in bp['caps']][::2]
      maximums = [item.get_xdata()[0] for item in bp['caps']][1::2]
      print(f'Minimums: {minimums}\n'
            f'Maximums: {maximums}')

      # Extract Q1 and Q3 from box patches
      for box in bp['boxes']:
            vertices = box.get_path().vertices
            q1_val = vertices[:, 0].min()  # Left edge of box
            q3_val = vertices[:, 0].max()  # Right edge of box
            print(f'Q1: {q1_val}\nQ3: {q3_val}')
            print(f"Below {q3_val}: {(col_df < q3_val).sum()}")
            print(f"Above {q3_val}: {(col_df >= q3_val).sum()}")

In [ ]:
plot_boxplot_with_stats("Regional nodes positive (1988+)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(col_df, bins=100, edgecolor='black')
axes[0].set_xlabel('Regional Nodes Positive')
axes[0].set_ylabel('Frequency')

survival_months = np.concatenate([y_train['time'], y_test['time']])
regional_nodes = pd.concat([X_train[col_name], X_test[col_name]])
axes[1].scatter(regional_nodes, survival_months, alpha=0.1, s=1)
axes[1].set_xlabel('Regional Nodes Positive')
axes[1].set_ylabel('Survival Months')

plt.tight_layout()
plt.show()


In [ ]:
pd.concat([X_train[col_name], X_test[col_name]])

In [ ]:
#How exactly does tumor size affect risk - Dependence plot
# Explicitly set interaction feature
# See how bone metastasis effect varies by another feature
shap.dependence_plot(
    "SEER Combined Mets at DX-bone (2010+)", 
    shap_values, 
    X_train,
    interaction_index="Regional nodes positive (1988+)"  # Color by this
)


In [ ]:
#Investigate time varying effects
shap_values = explainer.shap_values(X_train)

#Choose feature to investigate
feature_name = 'First malignant primary indicator'
feature_index = X_train.columns.get_loc(feature_name)

#Plot SHAP against survival time
plt.figure(figsize=(10, 6))
plt.scatter(y_train['time'], shap_values[:, feature_index], alpha=0.3)
plt.xlabel('Survival Time (months)')
plt.ylabel(f'SHAP value for {feature_name}')
plt.title(f'Time-varying effect of {feature_name}')
plt.axhline(y=0, color='r', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# Convert a comma-separated string to a list
string_value = "a,b,c,d"
list_value = string_value.split(',')
print(list_value)  # Output: ['a', 'b', 'c', 'd']